# 09 — TST Expanding 실행 전용

이 노트북은 `08_tst.ipynb`에서 static 결과를 만든 뒤, expanding protocol만 따로 실행하기 위한 노트북이다.

핵심 원칙:
- `results/dl_tuned_L.csv`에서 static 기준 best L을 읽어온다.
- `(regime, country, feature_set)` 한 cell이 끝날 때마다 partial CSV를 저장한다.
- Colab이 끊겨도 Google Drive partial 파일이 남아 있으면 다음 실행에서 완료된 cell은 skip한다.
- 전체 partial이 모이면 마지막 aggregation 셀에서 기존 `dl_results_{tier}.csv`에 `protocol=expanding` 결과를 append한다.

## 0. Colab 셋업

GitHub에서 이 노트북을 바로 열었다면 아래 셀을 먼저 실행한다. 이미 `/content/FINTEL`에서 작업 중이면 그대로 통과한다.

In [ ]:
from pathlib import Path

if not Path('/content/FINTEL').exists():
    !git clone -b TST_JH https://github.com/GHLee1016/FINTEL.git /content/FINTEL

%cd /content/FINTEL
!git fetch origin TST_JH
!git checkout TST_JH
!git pull --ff-only origin TST_JH
!pip install -q -r requirements.txt

import torch
print(f'torch {torch.__version__}, cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

if not Path('dataset_DL/L22').exists():
    print('dataset_DL not found. Building dataset_DL...')
    !jupyter nbconvert --to notebook --execute notebooks/04_build_dl_dataset.ipynb --inplace 2>&1 | tail -80
else:
    print('dataset_DL already exists.')

## 1. Imports / 설정

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.dl import load_one
from src.eval.metrics import evaluate
from src.eval.phases import iter_phases
from src.models.dl import TSTModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DEVICE = {DEVICE}')

In [ ]:
LS = [22, 60, 252]
REGIMES = config.REGIMES
COUNTRIES = config.COUNTRIES
TIERS = ['core', 'momentum', 'extended']

HP = dict(
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.2,
    activation='gelu',
    norm_first=True,
    max_len=300,
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=64,
    max_epochs=100,
    early_stop_patience=10,
    early_stop_min_delta=1e-5,
    early_stop_metric='qlike',
    lr_patience=5,
    lr_factor=0.5,
    lr_min=1e-6,
    grad_clip=1.0,
    seed=42,
)

REFIT_EVERY = 20

RESULTS_DIR = PROJECT_ROOT / 'results'
LOG_DIR = PROJECT_ROOT / 'log' / 'TST'
DL_MODELS_DIR = RESULTS_DIR / 'best_dl_models'
RESULTS_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
DL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('REGIMES =', REGIMES)
print('COUNTRIES =', COUNTRIES)
print('TIERS =', TIERS)
print('REFIT_EVERY =', REFIT_EVERY)

## 2. Google Drive partial 저장 위치

중간 저장은 Google Drive에 한다. Colab 런타임이 끊겨도 이 폴더의 partial CSV는 유지된다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FINTEL_TST_EXPANDING')
PARTIAL_DIR = DRIVE_ROOT / 'partials'
RESULT_PARTIAL_DIR = PARTIAL_DIR / 'results'
LOG_PARTIAL_DIR = PARTIAL_DIR / 'logs'
HISTORY_PARTIAL_DIR = PARTIAL_DIR / 'history'
CKPT_PARTIAL_DIR = DRIVE_ROOT / 'checkpoints'

for p in [RESULT_PARTIAL_DIR, LOG_PARTIAL_DIR, HISTORY_PARTIAL_DIR, CKPT_PARTIAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('partial save root:', DRIVE_ROOT)

## 3. Best L 로드 / helper

In [ ]:
tuning_path = RESULTS_DIR / 'dl_tuned_L.csv'
if not tuning_path.exists():
    raise FileNotFoundError('results/dl_tuned_L.csv가 필요하다. 08_tst.ipynb static 저장 셀을 먼저 실행해야 한다.')

tuning_df = pd.read_csv(tuning_path)
best_L_lookup = {
    (r.regime, r.country, r.feature_set): int(r.L)
    for r in tuning_df[tuning_df['is_best_L']].itertuples()
}
print(f'best L tuning loaded: {len(best_L_lookup)} cells')

def eval_phases(splits: dict, y_pred: np.ndarray, regime: str) -> list:
    test_meta = splits['test_meta'].copy()
    test_meta.index = pd.to_datetime(test_meta['prediction_date'])
    rows = []
    for phase_name, mask, _color in iter_phases(test_meta, regime):
        if mask.sum() == 0:
            continue
        metrics = evaluate(splits['test_y'][mask], y_pred[mask])
        rows.append({'phase': phase_name, **metrics})
    return rows

def cell_key(regime: str, country: str, tier: str) -> str:
    return f'{regime}_{country}_{tier}'

def result_path_for(regime: str, country: str, tier: str) -> Path:
    return RESULT_PARTIAL_DIR / f'{cell_key(regime, country, tier)}.csv'

def is_done(regime: str, country: str, tier: str) -> bool:
    return result_path_for(regime, country, tier).exists()

def completed_table() -> pd.DataFrame:
    rows = []
    for regime in REGIMES:
        for country in COUNTRIES:
            for tier in TIERS:
                rows.append({
                    'regime': regime,
                    'country': country,
                    'feature_set': tier,
                    'done': is_done(regime, country, tier),
                })
    return pd.DataFrame(rows)

display(completed_table().groupby('done').size().rename('count').to_frame())

## 4. 한 cell 실행 함수

한 `(regime, country, tier)`가 끝날 때마다 result/log/history/checkpoint를 Drive에 저장한다.

In [ ]:
def run_expanding_cell(regime: str, country: str, tier: str, overwrite: bool = False) -> None:
    key = cell_key(regime, country, tier)
    if is_done(regime, country, tier) and not overwrite:
        print(f'[SKIP] {key}: partial result already exists')
        return

    if (regime, country, tier) not in best_L_lookup:
        raise KeyError(f'best L missing for {key}')

    L = best_L_lookup[(regime, country, tier)]
    t0 = time.time()
    splits = load_one(regime, country, L, tier)

    history_X = np.concatenate([splits['train_X'], splits['valid_X']], axis=0)
    history_y = np.concatenate([splits['train_y'], splits['valid_y']], axis=0)

    model = None
    y_pred_list = []
    history_parts = []
    n_refits = 0

    for step in range(len(splits['test_X'])):
        if step % REFIT_EVERY == 0:
            n = len(history_X)
            n_val = max(1, int(0.15 * n))
            tr_X, tr_y = history_X[:-n_val], history_y[:-n_val]
            va_X, va_y = history_X[-n_val:], history_y[-n_val:]

            model = TSTModel(
                feature_cols=splits['feature_cols'],
                L=L,
                device=DEVICE,
                verbose=False,
                **HP,
            )
            model.fit(tr_X, tr_y, va_X, va_y)

            hist = model.history_df()
            hist.insert(0, 'protocol', 'expanding')
            hist.insert(0, 'L', L)
            hist.insert(0, 'feature_set', tier)
            hist.insert(0, 'country', country)
            hist.insert(0, 'regime', regime)
            hist.insert(0, 'refit_step', n_refits)
            history_parts.append(hist)
            n_refits += 1

        pred_step = float(model.predict(splits['test_X'][step:step + 1])[0])
        y_pred_list.append(pred_step)
        history_X = np.concatenate([history_X, splits['test_X'][step:step + 1]], axis=0)
        history_y = np.concatenate([history_y, splits['test_y'][step:step + 1]], axis=0)

    y_pred = np.clip(np.array(y_pred_list, dtype=np.float32), 1e-8, None)
    phase_rows = eval_phases(splits, y_pred, regime)

    result_rows = []
    for r in phase_rows:
        result_rows.append({
            'model': 'TST',
            'regime': regime,
            'country': country,
            'feature_set': tier,
            'L': L,
            'protocol': 'expanding',
            'phase': r['phase'],
            'RMSE': r['RMSE'],
            'MAE': r['MAE'],
            'QLIKE': r['QLIKE'],
            'RMSE_CV': r['RMSE_CV'],
            'best_val_loss': float(model.best_val_loss_),
            'epochs_used': int(model.epochs_used_),
            'n_features': len(splits['feature_cols']),
            'n_refits': n_refits,
            'is_best_L': True,
        })

    result_df = pd.DataFrame(result_rows)
    result_df.to_csv(result_path_for(regime, country, tier), index=False, encoding='utf-8-sig')

    log_df = pd.DataFrame({
        'date': splits['test_meta']['prediction_date'].values,
        'feature_set': tier,
        'L': L,
        'protocol': 'expanding',
        'RV_target': splits['test_y'],
        'RV_pred': y_pred,
    })
    log_df.to_csv(LOG_PARTIAL_DIR / f'log_TST_{key}.csv', index=False, encoding='utf-8-sig')

    if history_parts:
        pd.concat(history_parts, ignore_index=True).to_csv(
            HISTORY_PARTIAL_DIR / f'history_TST_{key}.csv',
            index=False,
            encoding='utf-8-sig',
        )

    ckpt_path = CKPT_PARTIAL_DIR / f'TST_{key}_L{L}_expanding.pt'
    model.save_checkpoint(
        ckpt_path,
        extra={'regime': regime, 'country': country, 'tier': tier, 'protocol': 'expanding', 'n_refits': n_refits},
    )

    full_rmse_cv = next(r['RMSE_CV'] for r in phase_rows if r['phase'] == 'Full Test')
    print(f'[DONE] {key:<22} L={L:>3} refits={n_refits:>3} RMSE_CV={full_rmse_cv:.4f} | {time.time() - t0:.1f}s')

## 5. 실행 방식 A — 하나만 테스트

처음에는 아래 한 줄만 돌려서 저장/skip이 잘 되는지 확인한다.

In [ ]:
# 예시 테스트. 이미 끝난 cell이면 자동 skip된다.
# run_expanding_cell('normal', 'US', 'core')

## 6. ?? ?? B ? 36? ?? ? ?? ??

?? 36? ?? ????? ???? ????. ? ?? ??? `(regime, country, feature_set)`? ???, ??? Google Drive? partial CSV? ????. ??? Colab? ??? ?? ? ???? ?? ?? ??? ???? ???? ?? ??? cell? `[SKIP]` ????.

### 01/36 ? `normal_US_core`

In [ ]:
run_expanding_cell('normal', 'US', 'core')


### 02/36 ? `normal_US_momentum`

In [ ]:
run_expanding_cell('normal', 'US', 'momentum')


### 03/36 ? `normal_US_extended`

In [ ]:
run_expanding_cell('normal', 'US', 'extended')


### 04/36 ? `normal_KR_core`

In [ ]:
run_expanding_cell('normal', 'KR', 'core')


### 05/36 ? `normal_KR_momentum`

In [ ]:
run_expanding_cell('normal', 'KR', 'momentum')


### 06/36 ? `normal_KR_extended`

In [ ]:
run_expanding_cell('normal', 'KR', 'extended')


### 07/36 ? `normal_JP_core`

In [ ]:
run_expanding_cell('normal', 'JP', 'core')


### 08/36 ? `normal_JP_momentum`

In [ ]:
run_expanding_cell('normal', 'JP', 'momentum')


### 09/36 ? `normal_JP_extended`

In [ ]:
run_expanding_cell('normal', 'JP', 'extended')


### 10/36 ? `911_US_core`

In [ ]:
run_expanding_cell('911', 'US', 'core')


### 11/36 ? `911_US_momentum`

In [ ]:
run_expanding_cell('911', 'US', 'momentum')


### 12/36 ? `911_US_extended`

In [ ]:
run_expanding_cell('911', 'US', 'extended')


### 13/36 ? `911_KR_core`

In [ ]:
run_expanding_cell('911', 'KR', 'core')


### 14/36 ? `911_KR_momentum`

In [ ]:
run_expanding_cell('911', 'KR', 'momentum')


### 15/36 ? `911_KR_extended`

In [ ]:
run_expanding_cell('911', 'KR', 'extended')


### 16/36 ? `911_JP_core`

In [ ]:
run_expanding_cell('911', 'JP', 'core')


### 17/36 ? `911_JP_momentum`

In [ ]:
run_expanding_cell('911', 'JP', 'momentum')


### 18/36 ? `911_JP_extended`

In [ ]:
run_expanding_cell('911', 'JP', 'extended')


### 19/36 ? `gfc_US_core`

In [ ]:
run_expanding_cell('gfc', 'US', 'core')


### 20/36 ? `gfc_US_momentum`

In [ ]:
run_expanding_cell('gfc', 'US', 'momentum')


### 21/36 ? `gfc_US_extended`

In [ ]:
run_expanding_cell('gfc', 'US', 'extended')


### 22/36 ? `gfc_KR_core`

In [ ]:
run_expanding_cell('gfc', 'KR', 'core')


### 23/36 ? `gfc_KR_momentum`

In [ ]:
run_expanding_cell('gfc', 'KR', 'momentum')


### 24/36 ? `gfc_KR_extended`

In [ ]:
run_expanding_cell('gfc', 'KR', 'extended')


### 25/36 ? `gfc_JP_core`

In [ ]:
run_expanding_cell('gfc', 'JP', 'core')


### 26/36 ? `gfc_JP_momentum`

In [ ]:
run_expanding_cell('gfc', 'JP', 'momentum')


### 27/36 ? `gfc_JP_extended`

In [ ]:
run_expanding_cell('gfc', 'JP', 'extended')


### 28/36 ? `covid_US_core`

In [ ]:
run_expanding_cell('covid', 'US', 'core')


### 29/36 ? `covid_US_momentum`

In [ ]:
run_expanding_cell('covid', 'US', 'momentum')


### 30/36 ? `covid_US_extended`

In [ ]:
run_expanding_cell('covid', 'US', 'extended')


### 31/36 ? `covid_KR_core`

In [ ]:
run_expanding_cell('covid', 'KR', 'core')


### 32/36 ? `covid_KR_momentum`

In [ ]:
run_expanding_cell('covid', 'KR', 'momentum')


### 33/36 ? `covid_KR_extended`

In [ ]:
run_expanding_cell('covid', 'KR', 'extended')


### 34/36 ? `covid_JP_core`

In [ ]:
run_expanding_cell('covid', 'JP', 'core')


### 35/36 ? `covid_JP_momentum`

In [ ]:
run_expanding_cell('covid', 'JP', 'momentum')


### 36/36 ? `covid_JP_extended`

In [ ]:
run_expanding_cell('covid', 'JP', 'extended')


## 7. 진행 상황 확인

In [ ]:
progress = completed_table()
display(progress)
print(f"done {int(progress.done.sum())}/{len(progress)}")

## 8. Partial 결과를 repo 결과 파일로 합치기

36개가 모두 끝난 뒤 실행한다. 끝나지 않은 cell이 있으면 경고만 띄우고, 완료된 partial만 합친다.

In [ ]:
progress = completed_table()
missing = progress[~progress.done]
if not missing.empty:
    print('WARNING: unfinished cells remain. Aggregating completed partials only:')
    display(missing)

partial_result_files = sorted(RESULT_PARTIAL_DIR.glob('*.csv'))
if not partial_result_files:
    raise FileNotFoundError(f'no partial result files under {RESULT_PARTIAL_DIR}')

df_exp = pd.concat([pd.read_csv(p) for p in partial_result_files], ignore_index=True)
print(f'expanding result rows loaded: {len(df_exp)} from {len(partial_result_files)} partial files')

for tier in TIERS:
    out_path = RESULTS_DIR / f'dl_results_{tier}.csv'
    existing = pd.read_csv(out_path) if out_path.exists() else pd.DataFrame()
    if not existing.empty and 'protocol' in existing.columns:
        existing = existing[existing['protocol'] != 'expanding']
    sub_exp = df_exp[df_exp['feature_set'] == tier]
    combined = pd.concat([existing, sub_exp], ignore_index=True)
    combined.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'{tier:>10}: static/other {len(existing)} + expanding {len(sub_exp)} -> {len(combined)} rows')

for regime in REGIMES:
    for country in COUNTRIES:
        log_parts = []
        for tier in TIERS:
            p = LOG_PARTIAL_DIR / f'log_TST_{regime}_{country}_{tier}.csv'
            if p.exists():
                log_parts.append(pd.read_csv(p))
        if not log_parts:
            continue
        out_path = LOG_DIR / f'log_TST_{regime}_{country}.csv'
        existing = pd.read_csv(out_path) if out_path.exists() else pd.DataFrame()
        if not existing.empty and 'protocol' in existing.columns:
            existing = existing[existing['protocol'] != 'expanding']
        combined = pd.concat([existing] + log_parts, ignore_index=True)
        combined.to_csv(out_path, index=False, encoding='utf-8-sig')

history_files = sorted(HISTORY_PARTIAL_DIR.glob('history_TST_*.csv'))
if history_files:
    history_exp = pd.concat([pd.read_csv(p) for p in history_files], ignore_index=True)
    history_exp.to_csv(LOG_DIR / 'training_history_expanding.csv', index=False, encoding='utf-8-sig')
    print(f'expanding history rows: {len(history_exp)}')

print('aggregation done')

## 9. CSV만 GitHub push

aggregation까지 완료한 뒤 실행한다. checkpoint `.pt`는 push하지 않는다.

In [ ]:
# from getpass import getpass
# token = getpass('GitHub token: ')
# !git remote set-url origin https://{token}@github.com/GHLee1016/FINTEL.git
# !git add results/dl_results_core.csv results/dl_results_momentum.csv results/dl_results_extended.csv
# !git add log/TST/log_TST_*.csv log/TST/training_history_expanding.csv
# !git commit -m "Add TST expanding results"
# !git push origin TST_JH